In [ ]:
import ast
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import StratifiedGroupKFold

from src.classif.Toxic_or_not import ToxicOrNot
from src.classif.mood_rf import MoodRF
from src.classif.sentiment_rf import SentimentRF
from src.classif.subtopic_rf import SubtopicRF
from src.trimodel_rf import TriModelRF
from src.utils import ensure_directory

pd.set_option('display.max_columns', None)

PROJECT_ROOT = Path.cwd().resolve().parent if str(Path.cwd()).endswith('notebooks') else Path.cwd()
DATA_PATH = PROJECT_ROOT / 'data' / 'processed' / 'data_cleaned_before_encode.csv'
MODELS_DIR = PROJECT_ROOT / 'models'
ensure_directory(MODELS_DIR)

print('Project root:', PROJECT_ROOT)
print('Dataset:', DATA_PATH)

if not DATA_PATH.exists():
    raise FileNotFoundError(f'Missing dataset: {DATA_PATH}')

df = pd.read_csv(DATA_PATH)
print('Rows:', len(df))
print('Columns:', list(df.columns))
print(df[['text_clean', 'toxicity', 'topic']].head())

In [ ]:
# prepare labels for toxicity and subtopic

def parse_topic_list(value):
    if pd.isna(value):
        return []
    try:
        parsed = ast.literal_eval(value)
        if isinstance(parsed, list):
            return [str(item).strip() for item in parsed if str(item).strip()]
    except Exception:
        pass
    return []

df['topic_list_parsed'] = df['topic_list'].apply(parse_topic_list)

# Binary toxicity target
X_text = df['text_clean'].fillna('').astype(str)
y_toxic = df['toxicity'].fillna(0).astype(int)

# These are proxy labels derived from toxicity, not independently annotated labels.
mood_labels = np.where(y_toxic == 1, 'angry', 'happy')
sentiment_labels = np.where(y_toxic == 1, 'negative', 'positive')

# Multi-label subtopics from topic_list
all_topics = sorted({topic for topics in df['topic_list_parsed'] for topic in topics})
print('Subtopics:', all_topics)

def make_subtopic_targets(row):
    topics = row['topic_list_parsed']
    return [1 if topic in topics else 0 for topic in all_topics]

subtopic_matrix = np.array(df.apply(make_subtopic_targets, axis=1).tolist(), dtype=int)
print('Subtopic matrix shape:', subtopic_matrix.shape)

# Keep identical cleaned texts together to prevent train/test leakage.
splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
train_indices, test_indices = next(
    splitter.split(X_text, y_toxic, groups=X_text)
)
X_train_text = X_text.iloc[train_indices]
X_test_text = X_text.iloc[test_indices]
ytox_train, ytox_test = y_toxic.iloc[train_indices], y_toxic.iloc[test_indices]
mood_train, mood_test = mood_labels[train_indices], mood_labels[test_indices]
sent_train, sent_test = sentiment_labels[train_indices], sentiment_labels[test_indices]
sub_train, sub_test = subtopic_matrix[train_indices], subtopic_matrix[test_indices]

# Fit TF-IDF on training text only to avoid learning from the held-out set.
vectorizer = TfidfVectorizer(max_features=2000, ngram_range=(1, 2), min_df=2)
X_train = vectorizer.fit_transform(X_train_text)
X_test = vectorizer.transform(X_test_text)
print('Feature matrix shapes:', X_train.shape, X_test.shape)
print('Train size:', X_train.shape[0], 'Test size:', X_test.shape[0])
print('Toxicity test distribution:', ytox_test.value_counts(normalize=True).round(3).to_dict())

In [ ]:
# Fit the project models
mood_model = MoodRF(random_state=42)
sentiment_model = SentimentRF(random_state=42)
subtopic_model = SubtopicRF(random_state=42)

mood_model.fit(X_train, mood_train)
sentiment_model.fit(X_train, sent_train)
subtopic_model.fit(X_train, sub_train, labels=all_topics)

print('Mood training accuracy (not a generalization score):', mood_model.model.score(X_train, mood_train))
print('Sentiment training accuracy (not a generalization score):', sentiment_model.model.score(X_train, sent_train))
print('Subtopic training exact-match accuracy:', subtopic_model.model.score(X_train, sub_train))

trimodel = TriModelRF(random_state=42)
trimodel.fit(X_train, mood_train, sent_train, sub_train, labels=all_topics)


def make_proto_features(prediction, mood_classes, sentiment_classes):
    mood_features = np.column_stack([
        prediction['mood'] == label for label in mood_classes
    ])
    sentiment_features = np.column_stack([
        prediction['sentiment'] == label for label in sentiment_classes
    ])
    return np.column_stack((mood_features, sentiment_features, prediction['subtopic']))


def make_rule_features(texts, classifier):
    rule_results = [classifier.classify(text) for text in texts]
    return np.column_stack((
        [int(result['toxic']) for result in rule_results],
        [len(result['matches']) for result in rule_results],
    ))


# Create out-of-fold proto-class columns to keep in-sample TriModel predictions
# from leaking target information into the final toxicity classifier.
proto_width = len(np.unique(mood_train)) + len(np.unique(sent_train)) + sub_train.shape[1]
train_proto_features = np.zeros((len(X_train_text), proto_width), dtype=float)
proto_splitter = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=123)
for fold_number, (fold_train, fold_valid) in enumerate(
    proto_splitter.split(X_train_text, ytox_train, groups=X_train_text)
):
    fold_trimodel = TriModelRF(
        mood_kwargs={'n_estimators': 100},
        sentiment_kwargs={'n_estimators': 100},
        subtopic_kwargs={'n_estimators': 100},
        random_state=42 + fold_number,
    )
    fold_trimodel.fit(
        X_train[fold_train],
        mood_train[fold_train],
        sent_train[fold_train],
        sub_train[fold_train],
        labels=all_topics,
    )
    fold_prediction = fold_trimodel.predict(X_train[fold_valid])
    train_proto_features[fold_valid] = make_proto_features(
        fold_prediction,
        fold_trimodel.mood_model.classes_,
        fold_trimodel.sentiment_model.classes_,
    )

# The rule features are added after the TriModel proto-class features.
toxicity_model = ToxicOrNot(random_state=42)
train_rule_features = make_rule_features(X_train_text, toxicity_model.rule_classifier)
toxicity_train_engineered = np.column_stack((train_proto_features, train_rule_features))
toxicity_model.fit(
    X_train_text,
    ytox_train,
    engineered_features=toxicity_train_engineered,
)

# Apply the full-training TriModel and the rule layer to held-out comments.
test_tri_prediction = trimodel.predict(X_test)
test_proto_features = make_proto_features(
    test_tri_prediction,
    trimodel.mood_model.classes_,
    trimodel.sentiment_model.classes_,
)
test_rule_features = make_rule_features(X_test_text, toxicity_model.rule_classifier)
toxicity_test_engineered = np.column_stack((test_proto_features, test_rule_features))

print('Tri-model fitted:', trimodel._is_fitted)
print('Toxicity SVM fitted:', toxicity_model._is_fitted)
print('Toxicity engineered feature shape:', toxicity_train_engineered.shape)
print('Proto-class columns use out-of-fold predictions for SVM training.')

# Save models and fitted vectorizers needed for future inputs.
import pickle

for name, model in {
    'mood': mood_model,
    'sentiment': sentiment_model,
    'subtopic': subtopic_model,
    'trimodel': trimodel,
    'vectorizer': vectorizer,
    'toxic_or_not': toxicity_model,
}.items():
    path = MODELS_DIR / f'{name}_model.pkl'
    with path.open('wb') as handle:
        pickle.dump(model, handle)
    print('Saved:', path)

In [ ]:
# Evaluate predictions against the actual held-out targets.
from sklearn.metrics import accuracy_score, brier_score_loss, f1_score, precision_score, recall_score, hamming_loss

mood_pred = mood_model.predict(X_test)
sentiment_pred = sentiment_model.predict(X_test)
subtopic_pred = subtopic_model.predict(X_test)
toxicity_test_predictions = toxicity_model.predict(
    X_test_text,
    engineered_features=toxicity_test_engineered,
)


def print_classification_metrics(name, y_true, y_pred):
    print(
        f'{name} held-out ->',
        'accuracy:', accuracy_score(y_true, y_pred),
        'macro precision:', precision_score(y_true, y_pred, average='macro', zero_division=0),
        'macro recall:', recall_score(y_true, y_pred, average='macro', zero_division=0),
        'macro F1:', f1_score(y_true, y_pred, average='macro', zero_division=0),
    )


print_classification_metrics('Mood proxy', mood_test, mood_pred)
print_classification_metrics('Sentiment proxy', sent_test, sentiment_pred)
print(
    'Subtopic held-out ->',
    'exact-match accuracy:', accuracy_score(sub_test, subtopic_pred),
    'micro F1:', f1_score(sub_test, subtopic_pred, average='micro', zero_division=0),
    'Hamming loss:', hamming_loss(sub_test, subtopic_pred),
)
print_classification_metrics('Toxicity SVM (primary)', ytox_test, toxicity_test_predictions['final_prediction'])
print_classification_metrics('Toxicity rules (signal only)', ytox_test, toxicity_test_predictions['rule_prediction'])
print_classification_metrics(
    'SVM OR rules (experimental)',
    ytox_test,
    toxicity_test_predictions['or_experimental_prediction'],
)
positive_class_index = list(toxicity_model.svm.classes_).index(1)
toxicity_positive_confidence = toxicity_test_predictions['class_probabilities'][:, positive_class_index]
print('Toxicity confidence Brier score:', brier_score_loss(ytox_test, toxicity_positive_confidence))
print('Rule/SVM disagreements sent for review:', int(toxicity_test_predictions['rule_review'].sum()))
print('Subtopic prediction shape:', subtopic_pred.shape)
print('Example prediction:', subtopic_pred[0])

trimodel_pred = trimodel.predict(X_test)
print('Trimodel keys:', list(trimodel_pred.keys()))
print('Trimodel mood sample:', trimodel_pred['mood'][:5])
print('Trimodel sentiment sample:', trimodel_pred['sentiment'][:5])
print('Trimodel subtopic sample shape:', trimodel_pred['subtopic'][:2].shape)

print(
    'Caution: mood and sentiment targets were derived directly from toxicity,',
    'so their held-out scores evaluate those proxy rules, not independently annotated mood/sentiment.',
)

# IndoToxic modeling

This notebook trains baseline classifiers on the cleaned dataset and reports metrics on a held-out split.

Important interpretation:
- Mood and sentiment are proxy labels derived from toxicity, not independent annotations.
- The TriModel produces proto-class columns first; its training-fold features for the final classifier are generated out-of-fold to reduce leakage.
- Exact-match lexicon hit/count features are appended after the TriModel features.
- `ToxicOrNot` is the final toxicity stage: its calibrated LinearSVC consumes text, proto-class columns, and rule features, and emits the toxicity decision and confidence.
- The SVM is the primary toxicity decision; lexicon hits that disagree with it are surfaced for review, not automatic overrides.
- Confidence is a calibrated class probability; inspect the held-out Brier score and error cases before relying on it.
- The SVM-OR-rules prediction is experimental; compare its held-out metrics before considering a policy change.
- The held-out metrics below compare predictions with held-out targets; training scores are not evidence of generalization.
- Subtopic labels come from `topic_list`; inspect per-label performance and real-world examples before relying on predictions.
- New political examples are exploratory predictions, not verified classifications.

In [ ]:
# Try generated Indonesian political posts and, optionally, recent public posts from X.
import os
from urllib.error import HTTPError, URLError
from urllib.parse import urlencode
from urllib.request import Request, urlopen

from src.preprocessing import clean_text


def fetch_recent_tagged_posts(max_results=10):
    bearer_token = os.environ.get('X_BEARER_TOKEN')
    if not bearer_token:
        print('X_BEARER_TOKEN is unset; skipping X API search.')
        return []

    query = '(#indonesia OR #sppg OR #mbg OR #viral OR #pemerintah OR #uu OR #dpr OR #jawa OR #korupsi OR #judi) lang:id -is:retweet'
    params = urlencode({
        'query': query,
        'max_results': max_results,
        'tweet.fields': 'created_at,lang',
    })
    request = Request(
        f'https://api.x.com/2/tweets/search/recent?{params}',
        headers={'Authorization': f'Bearer {bearer_token}'},
    )
    try:
        with urlopen(request, timeout=20) as response:
            payload = json.load(response)
        posts = [item['text'] for item in payload.get('data', []) if item.get('text')]
        print(f'Fetched {len(posts)} recent posts through the X API.')
        return posts
    except HTTPError as error:
        print(f'X API request failed with HTTP {error.code}; check API access and token permissions.')
    except URLError as error:
        print(f'X API request failed: {error.reason}')
    return []


generated_posts = [
    'Harga bahan pokok naik lagi, pemerintah perlu menjelaskan rencana pengendaliannya.',
    'Saya berharap kebijakan transportasi publik lebih berpihak kepada warga di luar Jakarta.',
    'DPR sebaiknya membuka proses pembahasan anggaran agar masyarakat bisa ikut mengawasi.',
    'Program bantuan ini membantu keluarga kami, semoga penyalurannya makin tepat sasaran.',
    'Debat pilkada tadi menarik, tetapi banyak janji kandidat yang belum disertai rencana jelas.',
]

x_posts = fetch_recent_tagged_posts()
input_posts = generated_posts + x_posts
input_sources = ['generated'] * len(generated_posts) + ['X recent search'] * len(x_posts)
cleaned_posts = [clean_text(post) for post in input_posts]
input_features = vectorizer.transform(cleaned_posts)
input_predictions = trimodel.transform(input_features)
input_proto_features = make_proto_features(
    input_predictions,
    trimodel.mood_model.classes_,
    trimodel.sentiment_model.classes_,
)
input_rule_features = make_rule_features(input_posts, toxicity_model.rule_classifier)
input_toxicity_engineered = np.column_stack((input_proto_features, input_rule_features))
input_toxicity_predictions = toxicity_model.predict(
    input_posts,
    engineered_features=input_toxicity_engineered,
)

feature_analyzer = vectorizer.build_analyzer()
vocabulary = vectorizer.vocabulary_


def vocabulary_coverage(text):
    features = feature_analyzer(text)
    if not features:
        return 0.0
    return sum(feature in vocabulary for feature in features) / len(features)


def top_subtopics(probabilities, labels, limit=3):
    ranked_indices = np.argsort(probabilities)[::-1][:limit]
    return [
        f'{labels[index]} ({probabilities[index]:.2f})'
        for index in ranked_indices
    ]


results = pd.DataFrame({
    'source': input_sources,
    'text': input_posts,
    'vocabulary_coverage': [vocabulary_coverage(post) for post in cleaned_posts],
    'predicted_mood_proxy': input_predictions['mood'],
    'predicted_sentiment_proxy': input_predictions['sentiment'],
    'toxicity_svm': np.where(input_toxicity_predictions['svm_prediction'] == 1, 'toxic', 'non_toxic'),
    'lexicon_rule_hit': input_toxicity_predictions['rule_prediction'].astype(bool),
    'toxicity_final': np.where(input_toxicity_predictions['final_prediction'] == 1, 'toxic', 'non_toxic'),
    'toxicity_confidence': np.round(input_toxicity_predictions['confidence'], 3),
    'rule_review': input_toxicity_predictions['rule_review'].astype(bool),
    'top_subtopics': [
        top_subtopics(row, input_predictions['subtopic_labels'])
        for row in input_predictions['subtopic_proba']
    ],
})
pd.set_option('display.max_colwidth', 180)
display(results)

print('TriModel proto-classes and exact-match rule features feed the final toxicity SVM.')
print('The final label follows the SVM; conflicting lexicon hits are review signals.')
print('Toxicity confidence is a calibrated predicted-class probability; held-out calibration is summarized by Brier score.')
print('Mood/sentiment remain toxicity-derived proxies, and subtopic scores are not calibrated probabilities.')
print('Low vocabulary coverage means the input contains features unseen during training.')
print('X results require authorized recent-search API access via X_BEARER_TOKEN.')

In [ ]:
# Apply transparent lexicon rules after the tri-model predictions.
from src.classif.rule_toxicity import RuleToxicityClassifier

rule_classifier = RuleToxicityClassifier()
rule_results = [rule_classifier.classify(text) for text in input_posts]

rule_results_table = pd.DataFrame({
    'source': input_sources,
    'text': input_posts,
    'rule_toxicity': [result['label'] for result in rule_results],
    'matched_expressions': [
        ', '.join(match['expression'] for match in result['matches'])
        for result in rule_results
    ],
    'matched_regions': [
        ', '.join(sorted({
            region
            for match in result['matches']
            for region in match['regions']
        }))
        for result in rule_results
    ],
})
display(rule_results_table)

print('A toxic result means at least one listed expression matched; this rule is not context-aware.')
print('A non_toxic result means no listed expression matched, not that the comment is verified harmless.')

In [ ]:
# Flag possible gambling promotions by the configured keyword + 2–3 digit pattern.
from src.classif.gambling_ad_detector import GamblingAdDetector

gambling_detector = GamblingAdDetector()
gambling_results = [gambling_detector.classify(text) for text in x_posts]
gambling_results_table = pd.DataFrame({
    'text': x_posts,
    'suspected_gambling_promo': [
        result['suspected_gambling_promo'] for result in gambling_results
    ],
    'matched_patterns': [
        ', '.join(f"{match['term']} + {match['number']}" for match in result['matches'])
        for result in gambling_results
    ],
    'verified_number_meanings': [
        '; '.join(
            f"{meaning['meaning']} ({meaning['meaning_status']}; sources: {', '.join(meaning['source_ids'])})"
            for match in result['matches']
            for meaning in match['meanings']
        )
        for result in gambling_results
    ],
})
display(gambling_results_table)

print('This is a heuristic candidate detector, not proof that a post is a gambling advertisement.')
print('Only source-backed number meanings marked verified in data/togelnumbers.json are displayed.')
print('Current prefixes cover the supplied examples; extend them only with reviewed terms.')